# Pytania z Ksiezyca - Mapa pytan sluchaczy Radia 357

### Czym jest ten notatnik?

Ten notatnik analizuje pytania zadawane przez sluchaczy w audycji **Pytania z Ksiezyca** w Radiu 357. Pytania zostaly zebrane do jednego pliku tekstowego (po jednym pytaniu w linii) i sa udostepnione publicznie w formie gist'a na GitHubie.

Notatnik bierze kazde pytanie i za pomoca **modelu jezykowego** zamienia je w wektor liczbowy (tzw. embedding), ktory "rozumie" znaczenie pytania. Dzieki temu mozna zobaczyc, ktore pytania sa do siebie podobne tresciowo, nawet jesli uzywaja roznych slow. Wynikiem jest **interaktywna mapa**, na ktorej kazde pytanie to punkt, a bliskosc punktow oznacza podobienstwo znaczeniowe.

W przeciwienstwie do notatnika LAMU, pytania z Ksiezyca **nie maja przypisanych kategorii tematycznych** - dlatego wszystkie punkty maja ten sam kolor, a wykres pokazuje wylacznie strukture semantyczna (gdzie skupiaja sie podobne pytania).

### Jak uruchomic ten notatnik?

1. Otworz strone [Google Colab](https://colab.research.google.com/)
2. Wybierz **Plik > Otworz notatnik > Przeslij** i wgraj ten plik (.ipynb)
3. W menu u gory wybierz **Srodowisko uruchomieniowe > Zmien typ srodowiska** i ustaw **GPU** (typ T4) - dzieki temu obliczenia beda szybsze, ale notatnik zadziala tez bez GPU
4. Kliknij **Srodowisko uruchomieniowe > Uruchom wszystko** (lub Ctrl+F9)
5. Poczekaj kilka minut - notatnik sam pobierze plik z pytaniami, zainstaluje biblioteki, zakoduje pytania i wygeneruje wykresy
6. Na koncu automatycznie pobierze sie plik HTML z interaktywnym wykresem, ktory mozna otworzyc w przegladarce offline

### Co robi kazdy krok?

| Krok | Co sie dzieje | Ile trwa |
|------|--------------|----------|
| 1. Instalacja bibliotek | Pobiera narzedzia potrzebne do analizy | ~1 min |
| 2. Pobranie danych | Sciaga plik z pytaniami z GitHub gist | kilka sekund |
| 3. Wczytanie pytan | Czyta plik linia po linii (kazda linia to jedno pytanie) | natychmiastowe |
| 4. Embeddingi | Model jezykowy zamienia kazde pytanie w wektor 768 liczb | ~2-3 min (GPU) dla ~1000 pytan |
| 5. Redukcja wymiarow | Algorytm UMAP sciaga 768 wymiarow do 2 (zeby mozna bylo narysowac) | ~30 sek |
| 6. Wykresy | Rysuje statyczny (matplotlib) i interaktywny (Plotly) wykres | natychmiastowe |
| 7. Eksport HTML | Zapisuje interaktywny wykres jako plik do pobrania | natychmiastowe |

### Czego potrzeba?

Niczego poza przegladarka internetowa. Caly kod uruchamia sie w chmurze Google (Colab), nie trzeba nic instalowac na swoim komputerze. Konto Google jest wymagane do korzystania z Colab.

In [ ]:
!pip install -q sentence-transformers umap-learn plotly requests

In [ ]:
import requests
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import umap
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'colab'

## 1. Pobranie i wczytanie pytan z pliku tekstowego

In [ ]:
DATA_URL = "https://gist.githubusercontent.com/BERENZ/a2f58c81f5983e8f6b9e5191be4bad8a/raw/0fee1aec9d5ec6769ebe6e20bdeb0600ee0be692/pytania-z-ksiezyca.txt"
DATA_PATH = "pytania-z-ksiezyca.txt"

response = requests.get(DATA_URL)
response.raise_for_status()
with open(DATA_PATH, "wb") as f:
    f.write(response.content)
print(f"Pobrano plik: {DATA_PATH} ({len(response.content) / 1024:.1f} KB)")

In [ ]:
# Wczytanie pytan - kazda niepusta linia to jedno pytanie
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_lines = f.readlines()

questions = [line.strip() for line in raw_lines if line.strip()]

df = pd.DataFrame({'question': questions})
print(f"Liczba pytan: {len(df)}")
print(f"\nPierwsze 5 pytan:")
df.head()

## 2. Kodowanie pytan za pomoca polskich embeddingów

In [ ]:
MODEL_NAME = "sdadas/st-polish-paraphrase-from-distilroberta"

model = SentenceTransformer(MODEL_NAME)
print(f"Model: {MODEL_NAME}")
print(f"Wymiar embeddingów: {model.get_sentence_embedding_dimension()}")

In [ ]:
embeddings = model.encode(df['question'].tolist(), show_progress_bar=True, batch_size=32)
print(f"Ksztalt macierzy embeddingów: {embeddings.shape}")

## 3. Redukcja wymiarowości (UMAP)

In [ ]:
reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='cosine',
    random_state=42
)

coords_2d = reducer.fit_transform(embeddings)
df['x'] = coords_2d[:, 0]
df['y'] = coords_2d[:, 1]
print(f"Redukcja wymiarów: {embeddings.shape[1]} -> 2")

## 4. Wizualizacja

### 4a. Wykres statyczny (matplotlib)

In [ ]:
POINT_COLOR = '#1f4e79'  # ciemny granat - kojarzy sie z noca i ksiezycem

fig, ax = plt.subplots(figsize=(14, 10))

ax.scatter(
    df['x'], df['y'],
    color=POINT_COLOR,
    s=20, alpha=0.5, edgecolors='white', linewidth=0.3
)

ax.set_title('Pytania z Ksiezyca - pytania sluchaczy Radia 357 w przestrzeni embeddingów\n'
             f'(model: {MODEL_NAME}, redukcja: UMAP, N = {len(df)})', fontsize=14)
ax.set_xlabel('UMAP 1')
ax.set_ylabel('UMAP 2')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('ksiezyc_questions_static.png', dpi=150, bbox_inches='tight')
plt.show()
print("Zapisano: ksiezyc_questions_static.png")

### 4b. Wykres interaktywny (Plotly)

In [ ]:
# Truncate long questions for hover display
df['question_short'] = df['question'].apply(
    lambda q: q[:80] + '...' if len(q) > 80 else q
)

fig = px.scatter(
    df, x='x', y='y',
    hover_data={'question': True, 'x': False, 'y': False,
                'question_short': False},
    title=f'Pytania z Ksiezyca - mapa pytan sluchaczy Radia 357 (N = {len(df)})',
    labels={'x': 'UMAP 1', 'y': 'UMAP 2'},
    width=900, height=700,
    color_discrete_sequence=[POINT_COLOR]
)

fig.update_traces(
    marker=dict(size=6, opacity=0.6, line=dict(width=0.3, color='white')),
    hovertemplate='%{customdata[0]}<extra></extra>'
)

fig.update_layout(
    font=dict(size=12),
    hoverlabel=dict(font_size=11),
    plot_bgcolor='white',
    xaxis=dict(showgrid=True, gridcolor='lightgray'),
    yaxis=dict(showgrid=True, gridcolor='lightgray'),
    showlegend=False
)

fig.show()

### 4c. Statystyki

In [ ]:
print("=" * 50)
print("Pytania z Ksiezyca - Podsumowanie")
print("=" * 50)
print(f"Calkowita liczba pytan: {len(df)}")
print(f"Srednia dlugosc pytania: {df['question'].str.len().mean():.1f} znakow")
print(f"Mediana dlugosci: {df['question'].str.len().median():.0f} znakow")
print(f"Najkrotsze pytanie: {df['question'].str.len().min()} znakow")
print(f"Najdluzsze pytanie: {df['question'].str.len().max()} znakow")
print(f"\nModel embeddingów: {MODEL_NAME}")
print(f"Wymiar embeddingów: {embeddings.shape[1]}")
print(f"Metoda redukcji: UMAP (n_neighbors=15, min_dist=0.1, metric=cosine)")

### 4d. Jak czytac ten wykres

---

Kazdy **punkt na wykresie to jedno pytanie** zadane przez sluchacza Radia 357 w audycji "Pytania z Ksiezyca". W odroznieniu od notatnika LAMU, tutaj pytania nie maja przypisanych kategorii tematycznych - dlatego wszystkie punkty maja ten sam kolor.

**Najwazniejsza zasada: im blizej siebie sa dwa punkty, tym bardziej podobne sa pytania** - nie w sensie literalnym (te same slowa), lecz w sensie znaczeniowym. Model jezykowy "rozumie" tresc pytania i umieszcza je w przestrzeni tak, ze pytania o podobnych rzeczach laduja obok siebie.

Interakcje z wykresem:
- **Najechanie myszka** na punkt - wyswietla pelna tresc pytania.
- **Zaznaczenie obszaru** (klik + przeciagniecie) - powiekszenie fragmentu wykresu. Podwojne klikniecie wraca do widoku calosciowego.
- **Przeciaganie** z wcisnietym Shift - przesuwanie wykresu.

Co warto zauwazyc:
- **Skupiska** punktow to obszary, gdzie sluchacze zadawali wiele podobnych pytan - np. o kosmos, zwierzeta, jedzenie, cialo czlowieka.
- **Rzadziej zaludnione obszary** to mniej typowe, oryginalne pytania.
- Osie X i Y (UMAP 1, UMAP 2) nie maja znaczenia fizycznego - to abstrakcyjne wymiary po redukcji z 768 wymiarow. Wazne sa wylacznie wzgledne odleglosci miedzy punktami.

## 5. Eksport do HTML (offline)

In [ ]:
HTML_PATH = "pytania_z_ksiezyca.html"
PNG_PATH = "ksiezyc_questions_static.png"

fig.write_html(HTML_PATH, include_plotlyjs=True, full_html=True)
print(f"Zapisano interaktywny wykres: {HTML_PATH}")
print(f"Zapisano statyczny PNG:      {PNG_PATH}")

# Download in Colab
try:
    from google.colab import files
    files.download(HTML_PATH)
    files.download(PNG_PATH)
    print("Pobieranie plikow...")
except ImportError:
    print(f"Otworz plik {HTML_PATH} w przegladarce, aby zobaczyc wykres offline.")